# IWI experiment · Evaluation 3: rural and urban clusters, on the whole continent and in every country

The third evaluation notebook. It reads the test predictions that 16 collected and splits them by the DHS flag of every cluster (`rural` of the DHS table: rural or urban). First on the **whole continent** (all countries pooled), then in **every country**: the scores of every model and input for rural and for urban clusters, and every input against the **baseline, Landsat + NL**, with the same model and start (pretrained with pretrained, scratch with scratch).

**Reading the numbers.** R² is computed within each subset, against the subset's own mean IWI: the R² of the rural clusters says how well a model ranks rural clusters among themselves. Rural and urban clusters differ in their spread of IWI (and the urban ones are fewer), so their R² are not on the same footing; RMSE, MAE and bias (predicted − observed, IWI points) are shown with them. The **gain over the baseline** (`dR2` = R²(input) − R²(Landsat + NL) on the same clusters, paired) is comparable between rural and urban, and section 3 tests whether it differs: `gain rural − gain urban` with its 95 % interval from the same bootstrap resamples. Intervals: `BOOTSTRAP` Poisson resamples of 03's spatial groups (a group of clusters closer than 9.5 km can hold rural and urban clusters, and keeps one weight for both); `q`: Benjamini–Hochberg over the models and inputs of a start, family and area (section 3; the baseline models on their own) and over the countries (section 5); `folds better`: test folds with at least `MIN_FOLD_N` of the subset's clusters in which the input has the higher R²; `verdict`: `better` or `worse` when the interval excludes 0.

**Sections:** 2. rural and urban scores on the whole continent; 3. every input against the baseline on the whole continent, rural and urban, and the difference of the gains; 4. rural and urban scores per country; 5. every input against the baseline per country, rural and urban; 6. figures per country; 7. every model against the **baseline models** VGG-16-BN and MLP (`BASELINE_MODELS`), rural and urban, on the whole continent and per country, with the same input and against the baseline model on Landsat + NL.

**Output** in `EVAL_DIR`: `rural_urban_continental.csv`, `rural_urban_continental_vs_baseline.csv`, `rural_urban_country_metrics.csv`, `rural_urban_country_vs_baseline.csv`, `rural_urban_country_consensus.csv`, `rural_urban_continental_vs_baseline_models.csv`, `rural_urban_country_vs_baseline_models.csv` (and a `_summary` of each over the ten models), and the plots `rural_urban_r2_<start>.png`, `rural_urban_vs_baseline_<start>.png`, `rural_urban_country_r2_baseline.png`, `rural_urban_country_consensus_<start>.png`, `rural_urban_country_<start>_<input>.png`, `rural_urban_country_<start>_<family>.png`, `rural_urban_vs_<baseline model>_<start>.png`, `rural_urban_country_vs_baseline_models_<start>_<baseline model>.png`.

**Running.** After 16, on a CPU (a few minutes).

# Import libraries

In [ ]:
import os
import re
import glob
import json
import time
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from IPython.display import display, Image

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
WORK_DIR  = '/path/to/data/'
FOLDS_CSV = os.path.join(WORK_DIR, 'train_clusters_folds.csv')   # written by 03_Spatial_Folds
PERIODS   = 'DMSP_VIIRS'   # the run folders of 06-15 that are read, <periods>_<start>...: both NL periods, as in 06-15
EVAL_DIR  = os.path.join(WORK_DIR, 'evaluation', PERIODS)       # 16 writes the collected predictions here, and
                                                                 # 16-18 their tables and plots

# ---- evaluation ----------------------------------------------------------------------------------
TARGET       = 'iwi'           # the label of FOLDS_CSV that every model predicts
FOLDS        = ['A', 'B', 'C', 'D', 'E']
BASELINE     = 'L+NL'          # the baseline input: the 7 Landsat bands and NL (12; the row 'base' of 04 and 05; the
                               # ridge on NL + Landsat features of the row 'landsat' of 09 and 10)
BOOTSTRAP    = 1000            # resamples of 03's spatial groups for the 95 % intervals (0: no intervals)
SEED         = 0               # the resamples: the same for every comparison, so all comparisons are paired
MIN_N        = 30              # an R2 needs at least this many test clusters (a country, its rural or urban clusters)
MIN_FOLD_N   = 10              # a test fold counts in 'folds better' with at least this many of the clusters
NOISE_MODELS = ['satclip_only']   # read no band, only the coordinates: their change between inputs is training noise
                                  # (left out of the summaries over models, shown as 'noise dR2')
BASELINE_MODELS = ['vgg16bn', 'mlp']   # the baseline models of 11-15 (added 2026-11-16: VGG-16-BN and an MLP on band
                                       # statistics, on the same inputs): in every table and plot of the models and
                                       # compared with every model (16, section 6), but left out of the summaries
                                       # over models (shown there on their own: 'vgg16bn dR2', 'mlp dR2'), so these
                                       # stay those of the ten models
SUMMARY_SKIP = NOISE_MODELS + BASELINE_MODELS   # the models that the summaries over models leave out

# ---- figures ------------------------------------------------------------------------------------------------------------
DETAIL_INPUTS = 'main'    # inputs of the models of 11-15 with figures of every model (sections 2, 3 and 6): 'main'
                          # (those of MAIN_INPUTS that were run: the default run of 11, 13, 14 and 15) or 'all'
AREAS = ['rural', 'urban']
VS_BASELINE_MODELS = True   # section 7: every model against the baseline models (VGG-16-BN, MLP), rural and urban

In [ ]:
# ---- the models and the inputs: names, labels and order ---------------------------------------------------------------
TEN_MODELS = ['resnet18', 'resnet34', 'resnet50', 'swin', 'unet', 'vit', 'satclip', 'satclip_only', 'mosaiks',
              'olmoearth']                                       # the models of 11-15, in their order
MODEL_LABEL = {'resnet18': 'ResNet-18', 'resnet34': 'ResNet-34', 'resnet50': 'ResNet-50', 'swin': 'Swin v2-B',
               'unet': 'U-Net', 'vit': 'ViT-S/16', 'satclip': 'ResNet-18 + SatCLIP', 'satclip_only': 'SatCLIP only',
               'mosaiks': 'MOSAIKS', 'olmoearth': 'OlmoEarth', 'vgg16bn': 'VGG-16-BN', 'mlp': 'MLP (band stats)',
               'vgg16': 'VGG-16 (04)',
               'resnet18_finetune': 'ResNet-18 finetune (05)', 'resnet18_frozen': 'ResNet-18 frozen (05)',
               'nl_resnet18': 'ResNet-18 on NL (06)', 'landsat_resnet18': 'ResNet-18 on Landsat (07)'}
FAMILY_ORDER = ['ten_models', 'table2', 'ridge', 'single']
FAMILY_LABEL = {'ten_models': 'the models of 11-15', 'table2': 'the Table 2 networks of 04 and 05',
                'ridge': 'the ridge on network features of 09 and 10', 'single': 'the NL or Landsat networks of 06-08'}
START_ORDER  = ['pretrained', 'scratch', 'mixed', 'unknown']
MAIN_INPUTS  = ['L+NL+wavelet', 'L+NL+DFT', 'L+NL+peak:pixel11/wavelet6', 'L+NL+TT:cyclic', 'L+NL+NDVI']
SINGLE_BANDS = ['LFC_min', 'LFC_max', 'LFC_mean', 'HFC_min', 'HFC_max', 'HFC_mean']   # the order of the band cache


def model_label(m):
    """Display name of a model key."""
    if m in MODEL_LABEL:
        return MODEL_LABEL[m]
    if m.startswith('ridge:'):
        return f'ridge, {m[6:]}'
    if m.startswith('landsat_vits16_'):
        return f'ViT-S/16 on Landsat (08, {m[15:]})'
    return m


OTHER_MODELS = ['vgg16', 'resnet18_finetune', 'resnet18_frozen', 'nl_resnet18', 'landsat_resnet18']


def model_order(m):
    """Sort key of a model: the ten models of 11-15 in their order and the baseline models, then those of 04-08, then
    the others by name."""
    if m in TEN_MODELS:
        return (0, TEN_MODELS.index(m), '')
    if m in BASELINE_MODELS:
        return (0, len(TEN_MODELS) + BASELINE_MODELS.index(m), '')
    if m in OTHER_MODELS:
        return (1, OTHER_MODELS.index(m), '')
    return (2, 0, m)


def combo_order(c):
    """Sort key of an input: the baseline, what 11, 13, 14 and 15 add to it, the rows of 04/05 and 09/10, then the
    inputs without NL (06-08)."""
    if c == BASELINE:
        return (0, 0, '')
    if not c.startswith(BASELINE + '+'):
        return (10, ['NL', 'L'].index(c), '') if c in ('NL', 'L') else (12, 0, c)
    tail = c[len(BASELINE) + 1:]
    if tail == 'wavelet':
        return (1, 0, '')
    if tail == 'DFT':
        return (2, 0, '')
    if tail.startswith('peak:'):
        align, _, inp = tail[5:].partition('/')
        aligns, inputs = ['pixel11', 'cluster11', 'calendar'], ['wavelet6', 'ndvi23', 'ndvi23_wavelet6']
        return (3, 10 * (inputs.index(inp) if inp in inputs else 3) + (aligns.index(align) if align in aligns else 3),
                tail)
    if tail.startswith('TT:'):
        return (4, {'cyclic': 0, 'none': 1, 'absolute': 2}.get(tail[3:], 3), tail)
    if tail == 'NDVI':
        return (5, 0, '')
    if tail in SINGLE_BANDS:
        return (6, SINGLE_BANDS.index(tail), '')
    return (7, 0, tail)


def short_combo(c):
    """Short name of an input: what it adds to Landsat + NL ('wavelet', 'DFT', 'peak pixel11/wavelet6', 'TT cyclic',
    'NDVI', 'HFC max', ...), and 'Landsat + NL' for the baseline."""
    if c == BASELINE:
        return 'Landsat + NL'
    tail = c[len(BASELINE) + 1:] if c.startswith(BASELINE + '+') else c
    return tail.replace('peak:', 'peak ').replace('TT:', 'TT ').replace('_', ' ')


def combo_label(c):
    """Long name of an input."""
    if c == BASELINE:
        return 'Landsat + NL (baseline)'
    if not c.startswith(BASELINE + '+'):
        return {'NL': 'NL only', 'L': 'Landsat only'}.get(c, c.replace('L+', 'Landsat + ').replace('_', ' '))
    tail = c[len(BASELINE) + 1:]
    if tail.startswith('peak:'):
        return f'Landsat + NL + peak-aligned NDVI, {tail[5:]} (14)'
    if tail.startswith('TT:'):
        return f'Landsat + NL + temporal transformer, {tail[3:]} (15)'
    names = {'wavelet': '6 wavelet bands', 'DFT': '5 DFT bands (13)', 'NDVI': '12 monthly NDVI values'}
    return 'Landsat + NL + ' + names.get(tail, tail.replace('_', ' '))


def sort_runs(frame):
    """Rows of a table with the columns start, family, model and combo in the order of START_ORDER, FAMILY_ORDER,
    model_order and combo_order."""
    keys = [(START_ORDER.index(s) if s in START_ORDER else 9, FAMILY_ORDER.index(f) if f in FAMILY_ORDER else 9,
             model_order(m), combo_order(c))
            for s, f, m, c in zip(frame['start'], frame['family'], frame['model'], frame['combo'])]
    return frame.iloc[sorted(range(len(frame)), key=keys.__getitem__)]

In [ ]:
SCORE_COLS = ['n', f'{TARGET} mean', f'{TARGET} sd', 'R2', 'r2', 'RMSE', 'MAE', 'bias']


def cell_scores(y, p, code, n_cells, min_n=None):
    """Scores of the prediction column p in every cell (code: the cell of every cluster, -1 = in none; clusters without
    a prediction are left out). Columns as SCORE_COLS: n, mean and sd of the observed IWI, R2 against the cell's own
    mean IWI (negative when the predictions miss the cell's level or spread), r2 (squared correlation of predicted and
    observed, which an offset does not lower; Yeh et al. 2020), RMSE, MAE and bias (mean of predicted - observed), in
    IWI points. R2 and r2 need min_n clusters (default MIN_N). Returns an array (n_cells, len(SCORE_COLS))."""
    min_n = MIN_N if min_n is None else min_n
    ok = np.isfinite(p) & (code >= 0)
    c, yy, pp = code[ok], y[ok], p[ok]
    e = pp - yy
    s = lambda w: np.bincount(c, weights=w, minlength=n_cells)[:n_cells]
    n = np.bincount(c, minlength=n_cells)[:n_cells].astype(float)
    sy, syy, sp, spp, spy = s(yy), s(yy * yy), s(pp), s(pp * pp), s(pp * yy)
    se, see, sae = s(e), s(e * e), s(np.abs(e))
    with np.errstate(invalid='ignore', divide='ignore'):
        my = sy / n
        sst = syy - sy * my                       # sum of squares around the cell's mean
        spp_c = spp - sp * sp / n
        cov = spy - sp * my
        enough = (n >= max(min_n, 2)) & (sst > 1e-9)
        r2 = np.where(enough, 1 - see / sst, np.nan)
        rr = np.where(enough & (spp_c > 1e-12), cov ** 2 / (sst * spp_c), np.nan)
        out = np.column_stack([n, my, np.sqrt(sst / (n - 1)), r2, rr, np.sqrt(see / n), sae / n, se / n])
    out[n == 0, 1:] = np.nan
    out[n < 2, 2] = np.nan
    return out


class Bootstrap:
    """Poisson bootstrap over 03's spatial groups (clusters closer than 9.5 km are one group, and a group lies in one
    fold): every resample gives every group a weight drawn from Poisson(1). The same weights serve every comparison, so
    all comparisons are paired, and a subset (a country, its rural clusters) is resampled with the groups it holds."""

    def __init__(self, unit, n_boot, seed):
        self.unit = np.asarray(unit, dtype=np.int64)
        self.n_boot = int(n_boot)
        n_units = int(self.unit.max()) + 1 if len(self.unit) else 0
        rng = np.random.default_rng(seed)
        self.W = rng.poisson(1.0, size=(self.n_boot, n_units)).astype(np.float64) if self.n_boot else None


def _pair_stats(T):
    """R2 of a and b, dR2 = R2(b) - R2(a), dRMSE and dMAE (b - a) from the sums T[..., :7]: n, sum y, sum y^2,
    SSE of a, SSE of b, sum |error of a|, sum |error of b|."""
    n, sy, syy, sa, sb, aa, ab = (T[..., k] for k in range(7))
    with np.errstate(invalid='ignore', divide='ignore'):
        sst = syy - sy * sy / n
        bad = (n < 2) | ~(sst > 1e-9)
        r2a = np.where(bad, np.nan, 1 - sa / sst)
        r2b = np.where(bad, np.nan, 1 - sb / sst)
        return r2a, r2b, r2b - r2a, np.sqrt(sb / n) - np.sqrt(sa / n), (ab - aa) / n


PAIR_COLS = ['n', 'R2 a', 'R2 b', 'dR2', 'dR2 low', 'dR2 high', 'P(dR2 > 0)', 'p', 'dRMSE', 'dRMSE low', 'dRMSE high',
             'dMAE', 'folds better', 'folds compared']


def compare(y, pa, pb, cells, boot, fold_code=None, draws=False):
    """Prediction column b against a on the clusters where both have a prediction, in every cell (name -> positions of
    its clusters). Per cell: n, R2 of a and b, dR2 = R2(b) - R2(a) (a and b share the clusters, so this is their
    difference in squared error over the cell's sum of squares), its 95 % bootstrap interval, P(dR2 > 0) and the
    two-sided p of the resamples, dRMSE and dMAE (b - a in IWI points; negative = b better), and in how many test folds
    b has the higher R2 (of the folds with at least MIN_FOLD_N of the cell's clusters). A cell with fewer than MIN_N
    common clusters gets n only. Returns {cell: dict with the keys of PAIR_COLS}; draws=True adds 'draws', the dR2 of
    every resample (the same resamples for every cell and comparison, so differences of draws are paired)."""
    both = np.isfinite(pa) & np.isfinite(pb)
    ea, eb = y - pa, y - pb
    Q = np.column_stack([np.ones(len(y)), y, y * y, ea * ea, eb * eb, np.abs(ea), np.abs(eb)])
    out = {}
    for name, pos in cells.items():
        pos = pos[both[pos]]
        rec = {'n': int(len(pos))}
        out[name] = rec
        if len(pos) < MIN_N:
            continue
        r2a, r2b, d, drmse, dmae = (float(v) for v in _pair_stats(Q[pos].sum(axis=0)))
        rec.update({'R2 a': r2a, 'R2 b': r2b, 'dR2': d, 'dRMSE': drmse, 'dMAE': dmae})
        if boot is not None and boot.n_boot:
            u, inv = np.unique(boot.unit[pos], return_inverse=True)
            inv = inv.ravel()
            S = np.column_stack([np.bincount(inv, weights=Q[pos, k], minlength=len(u)) for k in range(7)])
            _, _, bd, brmse, _ = _pair_stats(boot.W[:, u] @ S)
            ok = np.isfinite(bd)
            if draws:
                rec['draws'] = bd
            if ok.sum() >= 10:
                b = bd[ok]
                rec.update({'dR2 low': float(np.quantile(b, 0.025)), 'dR2 high': float(np.quantile(b, 0.975)),
                            'P(dR2 > 0)': float((b > 0).mean()),
                            'p': float(min(1.0, 2 * (1 + min((b <= 0).sum(), (b >= 0).sum())) / (len(b) + 1))),
                            'dRMSE low': float(np.nanquantile(brmse[ok], 0.025)),
                            'dRMSE high': float(np.nanquantile(brmse[ok], 0.975))})
        if fold_code is not None:
            f = fold_code[pos]
            keep = f >= 0
            Sf = np.column_stack([np.bincount(f[keep], weights=Q[pos[keep], k], minlength=len(FOLDS))[:len(FOLDS)]
                                  for k in range(7)])
            use = Sf[:, 0] >= MIN_FOLD_N
            fd = _pair_stats(Sf[use])[2]
            rec['folds better'] = int((fd > 0).sum())
            rec['folds compared'] = int(np.isfinite(fd).sum())
    return out


def bh(p):
    """Benjamini-Hochberg q-values of the p-values p (NaN stays NaN)."""
    p = np.asarray(p, dtype=float)
    q = np.full(len(p), np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if len(ok):
        order = ok[np.argsort(p[ok], kind='stable')]
        ranked = p[order] * len(ok) / np.arange(1, len(ok) + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def add_q(frame, keys, where=None, split=None):
    """Column 'q': the Benjamini-Hochberg q-values of 'p' within every group of keys (rows of `where` only); `split`
    (one label per row, e.g. baseline_split) divides the groups further."""
    frame = frame.copy()
    frame['q'] = np.nan
    rows = frame.index if where is None else frame.index[np.asarray(where, bool)]
    by = list(keys)
    if split is not None:
        frame['_split'] = np.asarray(split)
        by.append('_split')
    for _, g in frame.loc[rows].groupby(by, sort=False):
        frame.loc[g.index, 'q'] = bh(g['p'].to_numpy(float))
    return frame.drop(columns='_split') if split is not None else frame


def baseline_split(frame):
    """'baseline' for the rows of a baseline model (BASELINE_MODELS; in a table of pairs: either model), 'main' for all
    others: add_q(..., split=baseline_split(frame)) gives the comparisons with a baseline model their own q-values, so
    those of the other models stay the same whether or not the baselines have run."""
    is_base = frame['model'].isin(BASELINE_MODELS).to_numpy()
    if 'other model' in frame:
        is_base = is_base | frame['other model'].isin(BASELINE_MODELS).to_numpy()
    return np.where(is_base, 'baseline', 'main')


def verdict(low, high):
    """'better' when the 95 % interval of a difference lies above 0, 'worse' below 0, 'n.s.' when it holds 0."""
    low, high = np.asarray(low, dtype=float), np.asarray(high, dtype=float)
    return np.where(low > 0, 'better', np.where(high < 0, 'worse', np.where(np.isfinite(low), 'n.s.', '')))


def fmt_delta(v, digits=3):
    """A signed difference without the leading zero: 0.0123 -> '+.012', -1.5 -> '-1.500', NaN -> '–'."""
    if v is None or not np.isfinite(v):
        return '–'
    return re.sub(r'^([+-])0\.', r'\1.', f'{v:+.{digits}f}')


def fmt_interval(d, low, high, digits=3):
    """'+.012 [+.004, +.020]' (without the interval when there is none)."""
    if d is None or not np.isfinite(d):
        return '–'
    if low is None or not np.isfinite(low):
        return fmt_delta(d, digits)
    return f'{fmt_delta(d, digits)} [{fmt_delta(low, digits)}, {fmt_delta(high, digits)}]'


def write_csv(frame, path, **kw):
    """Write a CSV file (gzip for .gz) under a temporary name, then rename it, so nobody reads half a file."""
    tmp = f'{path}.{uuid.uuid4().hex[:8]}.tmp'
    if path.endswith('.gz'):
        kw.setdefault('compression', 'gzip')
    frame.to_csv(tmp, index=False, **kw)
    os.replace(tmp, path)

In [ ]:
# Colors of the reference palette of the data-viz guide: chart ink, a blue -> red diverging scale with a gray midpoint
# for differences (blue: better than the reference), a one-hue blue scale for R2, and categorical slots 1 and 2 (blue,
# orange; checked for colour-vision deficiency) for rural and urban, with their own marker shapes.
INK, SECONDARY, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
SERIES_1    = '#2a78d6'                                    # categorical slot 1: a single series
AREA_COLOR  = {'rural': '#2a78d6', 'urban': '#eb6834'}
AREA_MARKER = {'rural': 'o', 'urban': 's'}
SEQUENTIAL  = LinearSegmentedColormap.from_list('r2_blue', ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf',
                                                            '#184f95', '#0d366b'])
DIVERGING   = LinearSegmentedColormap.from_list('delta', ['#e34948', '#f0efec', '#2a78d6'])
for _cmap in (SEQUENTIAL, DIVERGING):
    _cmap.set_bad(SURFACE)
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
                     'axes.edgecolor': AXIS, 'axes.labelcolor': SECONDARY, 'axes.titlecolor': INK,
                     'axes.titlesize': 10, 'axes.titlelocation': 'left', 'axes.labelsize': 9, 'font.size': 9,
                     'xtick.color': SECONDARY, 'ytick.color': SECONDARY, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'text.color': INK, 'legend.frameon': False, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False, 'grid.color': GRID,
                     'grid.linewidth': 0.8})


def text_color(rgba):
    """White on dark cells, ink on light ones."""
    r, g, b = rgba[:3]
    return 'white' if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.45 else INK


def delta_norm(values, floor=0.01, q=0.95):
    """A symmetric scale for differences: +-(the q quantile of |values|, at least floor), rounded up to 0.005."""
    v = np.abs(np.asarray(values, dtype=float))
    v = v[np.isfinite(v)]
    top = max(floor, float(np.quantile(v, q)) if len(v) else floor)
    top = np.ceil(top / 0.005) * 0.005
    return Normalize(-top, top)


def heatmap(ax, values, row_labels, col_labels, cmap, norm, labels=None, marks=None, bold=None, fontsize=6.5):
    """Coloured cells (rows x cols) with a thin surface gap. labels: text per cell (None: none); marks: cells that get a
    small dot (a significant difference, when there is no text); bold: cells whose text is bold. NaN cells get a muted
    dash."""
    values = np.asarray(values, dtype=float)
    im = ax.imshow(np.ma.masked_invalid(values), cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=90)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.set_xticks(np.arange(-0.5, len(col_labels)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_labels)), minor=True)
    ax.grid(which='minor', color=SURFACE, linewidth=1.5)
    ax.grid(which='major', visible=False)
    ax.tick_params(which='both', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            if not np.isfinite(v):
                ax.text(j, i, '–', ha='center', va='center', fontsize=fontsize, color=MUTED)
                continue
            color = text_color(cmap(norm(v)))
            if labels is not None and labels[i][j]:
                ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=fontsize, color=color,
                        fontweight='bold' if bold is not None and bold[i][j] else 'normal')
            elif marks is not None and marks[i][j]:
                ax.plot(j, i, marker='o', markersize=2.6, color=color, linestyle='none')
    return im


def forest(ax, y, d, low, high, color, marker='o', label=None):
    """Differences d with their 95 % intervals at the heights y: a filled marker where the interval excludes 0, an open
    one where it holds 0. Returns a handle for the legend."""
    d, low, high, y = (np.asarray(v, dtype=float) for v in (d, low, high, y))
    has = np.isfinite(low) & np.isfinite(high)
    ax.hlines(y[has], low[has], high[has], color=color, linewidth=1.4)
    sig = has & ((low > 0) | (high < 0))
    ok = np.isfinite(d)
    ax.scatter(d[ok & sig], y[ok & sig], s=34, marker=marker, color=color, edgecolors=SURFACE, linewidths=1.0,
               zorder=3)
    ax.scatter(d[ok & ~sig], y[ok & ~sig], s=34, marker=marker, facecolors=SURFACE, edgecolors=color,
               linewidths=1.3, zorder=3)
    return Line2D([], [], marker=marker, linestyle='none', color=color, markersize=6, label=label)


FILLED_OPEN = [Line2D([], [], marker='o', linestyle='none', color=SECONDARY, markersize=6,
                      label='filled: the 95 % interval excludes 0'),
               Line2D([], [], marker='o', linestyle='none', markerfacecolor=SURFACE, markeredgecolor=SECONDARY,
                      markersize=6, label='open: it holds 0')]


def save_figure(fig, name):
    """Save a figure as EVAL_DIR/<name>.png (written under a temporary name, then renamed) and show that file, so the
    notebook shows exactly the saved plot."""
    path = os.path.join(EVAL_DIR, f'{name}.png')
    fig.savefig(path + '.tmp.png', dpi=150, bbox_inches='tight')
    os.replace(path + '.tmp.png', path)
    plt.close(fig)
    display(Image(filename=path))
    print('->', path)


def grid_of(n, ncols):
    """Rows and columns of a grid of n panels with at most ncols columns, as even as possible (4 panels: 2 x 2)."""
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    return nrows, int(np.ceil(n / nrows))

In [ ]:
def consensus(table, keys, extras=True):
    """Over the models of every group of keys (without SUMMARY_SKIP: the noise and the baseline models): models, median
    and mean dR2, models with dR2 > 0, models significantly better / worse (interval above / below 0), the model with
    the largest gain; the mean dR2 of the noise models (training noise) and the dR2 of every baseline model. With
    extras=False (the comparisons with a baseline model) the model with the smallest dR2 instead of the last two."""
    out = []
    baselines = [b for b in BASELINE_MODELS if b in set(table['model'])]
    for k, g in table.groupby(keys, sort=False):
        real = g[~g['model'].isin(SUMMARY_SKIP) & g['dR2'].notna()]
        noise = g[g['model'].isin(NOISE_MODELS) & g['dR2'].notna()]
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        rec.update({'n': int(g['n'].max()), 'models': len(real),
                    'median dR2': real['dR2'].median() if len(real) else np.nan,
                    'mean dR2': real['dR2'].mean() if len(real) else np.nan,
                    'models better': int((real['dR2'] > 0).sum()),
                    'sig. better': int((real['verdict'] == 'better').sum()),
                    'sig. worse': int((real['verdict'] == 'worse').sum()),
                    'best model': model_label(real.loc[real['dR2'].idxmax(), 'model']) if len(real) else '',
                    'best dR2': real['dR2'].max() if len(real) else np.nan})
        if extras:
            rec.update({'noise dR2': noise['dR2'].mean() if len(noise) else np.nan,
                        **{f'{b} dR2': g.loc[g['model'] == b, 'dR2'].mean() for b in baselines}})
        else:
            rec.update({'worst model': model_label(real.loc[real['dR2'].idxmin(), 'model']) if len(real) else '',
                        'worst dR2': real['dR2'].min() if len(real) else np.nan})
        out.append(rec)
    out = pd.DataFrame(out)
    return sort_runs(out.assign(model='')).drop(columns='model').reset_index(drop=True) if len(out) else out


def consensus_cell(r):
    """'+.012 (8/9 ▲3 ▼1)': median dR2, models better / models, significantly better (▲) and worse (▼)."""
    if not r['models'] or not np.isfinite(r['median dR2']):
        return ''
    return (f"{fmt_delta(r['median dR2'])} ({r['models better']}/{r['models']}"
            + (f" ▲{r['sig. better']}" if r['sig. better'] else '') + (f" ▼{r['sig. worse']}" if r['sig. worse'] else '')
            + ')')

In [ ]:
# ---- every model of 11-15 against the baseline models (BASELINE_MODELS: VGG-16-BN and the MLP on band statistics) ------
BM_AGAINST = ['same input', 'Landsat + NL']   # the run of the baseline model that a run is compared with: the one with
                                              # the same input (what the architecture adds over the baseline model) and
                                              # the one with Landsat + NL (what the architecture and the added bands add
                                              # together over the baseline model on the baseline input)
BM_COLS = ['start', 'family', 'model', 'combo', 'baseline model', 'against', 'baseline run']


def baseline_model_pairs(frame):
    """(run, run of the baseline model, baseline model, against) for every run of the models of 11-15 in frame and every
    baseline model with a run of the same start: against its run with the same input ('same input'; not for the
    baseline model itself) and against its run with Landsat + NL ('Landsat + NL'; for the baseline model itself this is
    its own gain from the input, as in the comparisons with Landsat + NL)."""
    t = frame[frame['family'] == 'ten_models']
    rid = {(r.model, r.combo, r.start): r.run_id for r in t.itertuples()}
    out = []
    for r in t.itertuples():
        for b in BASELINE_MODELS:
            same, base = rid.get((b, r.combo, r.start)), rid.get((b, BASELINE, r.start))
            if same is not None and r.model != b:
                out.append((r.run_id, same, b, 'same input'))
            if base is not None and base != r.run_id:
                out.append((r.run_id, base, b, 'Landsat + NL'))
    return out


def sort_bm(frame):
    """Rows in the order of start, baseline model, reference (against), model and input; rows that share these (the
    countries, the areas) keep their order."""
    keys = [(START_ORDER.index(s) if s in START_ORDER else 9, BASELINE_MODELS.index(b) if b in BASELINE_MODELS else 9,
             BM_AGAINST.index(a) if a in BM_AGAINST else 9, model_order(m), combo_order(c))
            for s, b, a, m, c in zip(frame['start'], frame['baseline model'], frame['against'], frame['model'],
                                     frame['combo'])]
    return frame.iloc[sorted(range(len(frame)), key=keys.__getitem__)].reset_index(drop=True)


def bm_label(b, against):
    """'VGG-16-BN, same input' or 'VGG-16-BN on Landsat + NL': a baseline model and the run it is compared with."""
    return f'{model_label(b)}, same input' if against == 'same input' else f'{model_label(b)} on Landsat + NL'


def bm_summary(table, keys):
    """consensus() of a table of comparisons with the baseline models over the ten models (without SUMMARY_SKIP), for
    every start, baseline model, reference and the other keys, in the order of sort_bm."""
    if not len(table):
        return pd.DataFrame(columns=['start', 'family', 'baseline model', 'against'] + list(keys) +
                            ['n', 'models', 'median dR2', 'mean dR2', 'models better', 'sig. better', 'sig. worse',
                             'best model', 'best dR2', 'worst model', 'worst dR2'])
    out = consensus(table.assign(family='ten_models'), ['start', 'family', 'baseline model', 'against'] + list(keys),
                    extras=False)
    return sort_bm(out.assign(model='')).drop(columns='model')

# 1. The collected test predictions

In [ ]:
RUNS_CSV = os.path.join(EVAL_DIR, 'eval_runs.csv')
PRED_CSV = os.path.join(EVAL_DIR, 'eval_test_predictions.csv.gz')
assert os.path.exists(RUNS_CSV) and os.path.exists(PRED_CSV), \
    f'{RUNS_CSV} or {PRED_CSV} is missing: run 16_Evaluate_All_Models first (same EVAL_DIR)'

runs = pd.read_csv(RUNS_CSV, dtype={'folds': str, 'notebook': str}, keep_default_na=False,
                   na_values={'n_test': [''], 'n_folds': ['']})
runs['used'] = runs['used'].astype(str).str.lower().eq('true')
runs = sort_runs(runs[runs['used']]).set_index('run_id', drop=False)
assert len(runs), f'no used run in {RUNS_CSV}: run 16 again when runs of 04-15 have finished'
wide = pd.read_csv(PRED_CSV, dtype={'cluster_id': str, 'country': str, 'area': str, 'fold': str, 'nl_sensor': str})
missing = [r for r in runs['run_id'] if r not in wide.columns]
assert not missing, f'runs of eval_runs.csv without predictions in {PRED_CSV}: {missing[:5]}'

N = len(wide)
Y = wide[TARGET].to_numpy(dtype=float)
P = {r: wide[r].to_numpy(dtype=float) for r in runs['run_id']}
COUNTRY = wide['country'].fillna('').astype(str).to_numpy()
AREA = wide['area'].fillna('').astype(str).to_numpy()
COUNTRIES = sorted(set(COUNTRY) - {''})
COUNTRY_CODE = pd.Categorical(COUNTRY, categories=COUNTRIES).codes.astype(np.int64)
FOLD_CODE = pd.Categorical(wide['fold'].fillna('').astype(str), categories=FOLDS).codes.astype(np.int64)
BOOT = Bootstrap(pd.factorize(wide['group'])[0], BOOTSTRAP, SEED)
CONTINENT = f'all {len(COUNTRIES)} countries'
N_COUNTRY = pd.Series(COUNTRY).value_counts()
os.makedirs(EVAL_DIR, exist_ok=True)

print(f'{len(runs)} runs with test predictions for {N} clusters in {len(COUNTRIES)} countries ({RUNS_CSV})')
print('runs per start and family:')
display(runs.pivot_table(index='start', columns='family', values='run_id', aggfunc='count', fill_value=0)
        .reindex(index=[s for s in START_ORDER if s in set(runs['start'])],
                 columns=[f for f in FAMILY_ORDER if f in set(runs['family'])]))
print(f'bootstrap: {BOOTSTRAP} resamples of {BOOT.W.shape[1] if BOOT.n_boot else 0} spatial groups')


def baseline_pairs(frame):
    """(run, the run of its baseline input) for every run of frame whose input is not the baseline and whose family,
    model and start have a baseline run."""
    base = {(r.family, r.model, r.start): r.run_id for r in frame.itertuples() if r.combo == BASELINE}
    return [(r.run_id, base[(r.family, r.model, r.start)]) for r in frame.itertuples()
            if r.combo != BASELINE and (r.family, r.model, r.start) in base]


def run_info(run_id):
    """The columns of eval_runs.csv that describe a run."""
    r = runs.loc[run_id]
    return {'start': r['start'], 'family': r['family'], 'model': r['model'], 'combo': r['combo'],
            'notebook': r['notebook']}


def country_label(c):
    """'Nigeria (3759)': the country and its number of test clusters."""
    return f'{c} ({N_COUNTRY.get(c, 0)})'

flags = pd.Series(AREA).replace('', 'no flag').value_counts()
print('clusters per DHS flag:', flags.to_dict())
assert {'rural', 'urban'} & set(flags.index), 'the collected predictions have no rural or urban flag (column rural of 03)'
print(pd.crosstab(pd.Series(COUNTRY, name='country'), pd.Series(AREA, name='area').replace('', 'no flag'),
                  margins=True).to_string())


def detail_inputs(combos):
    """The inputs of the models of 11-15 with figures of every model: DETAIL_INPUTS of what was run."""
    combos = sorted(set(combos) - {BASELINE}, key=combo_order)
    if DETAIL_INPUTS == 'all':
        return combos
    return [c for c in MAIN_INPUTS if c in combos] or combos

# 2. Rural and urban clusters on the whole continent

The scores of every run on its rural, its urban and all its test clusters (`rural_urban_continental.csv`). The tables show the R² and RMSE of every model and input for rural and urban clusters, one table per start (pretrained with pretrained, scratch with scratch); the plot shows the R² of the models of 11–15 (circle: rural, square: urban).

In [ ]:
ZERO = np.zeros(N, dtype=np.int64)
rows = []
for r in runs.itertuples():
    for area in ['all'] + AREAS:
        code = ZERO if area == 'all' else np.where(AREA == area, 0, -1)
        v = cell_scores(Y, P[r.run_id], code, 1)[0]
        if v[0] > 0:
            rows.append({**run_info(r.run_id), 'area': area, **dict(zip(SCORE_COLS, v)), 'run_id': r.run_id})
cont = pd.DataFrame(rows)
write_csv(cont, os.path.join(EVAL_DIR, 'rural_urban_continental.csv'))
print(f'{len(cont)} rows (run x area) ->', os.path.join(EVAL_DIR, 'rural_urban_continental.csv'))


def area_table(t):
    """One row per run: n, R2, RMSE and bias of its rural and urban clusters, and the R2 of all."""
    w = t.pivot_table(index=['family', 'model', 'combo'], columns='area', values=['n', 'R2', 'RMSE', 'bias'],
                      aggfunc='first')
    out = pd.DataFrame(index=w.index)
    for col in ['n', 'R2', 'RMSE', 'bias']:
        for area in AREAS:
            if (col, area) in w.columns:
                out[f'{col} {area}'] = w[(col, area)]
    if ('R2', 'all') in w.columns:
        out['R2 all'] = w[('R2', 'all')]
    if {'R2 rural', 'R2 urban'} <= set(out.columns):
        out['R2 urban - rural'] = out['R2 urban'] - out['R2 rural']
    for area in AREAS:
        if f'n {area}' in out:
            out[f'n {area}'] = out[f'n {area}'].astype('Int64')
    out = out.reset_index()
    out = sort_runs(out.assign(start='')).drop(columns='start')
    out['model'] = out['model'].map(model_label)
    out['combo'] = out['combo'].map(short_combo)
    return out.rename(columns={'combo': 'input'}).reset_index(drop=True)


for start in [s for s in START_ORDER if s in set(cont['start'])]:
    for cl, fams in (('all clusters', ['ten_models', 'ridge', 'single']), ('VIIRS clusters (Table 2)', ['table2'])):
        t = cont[(cont['start'] == start) & cont['family'].isin(fams)]
        if len(t):
            print(f'\n{start.upper()}, {cl}: rural and urban test clusters on the whole continent')
            display(area_table(t).round(3))

In [ ]:
def r2_area_figure(start):
    t = cont[(cont['start'] == start) & (cont['family'] == 'ten_models') & cont['area'].isin(AREAS)]
    if not len(t):
        return
    combos = [BASELINE] + detail_inputs(t['combo']) if BASELINE in set(t['combo']) else detail_inputs(t['combo'])
    models = sorted(set(t['model']), key=model_order)
    yy = np.arange(len(models))[::-1]
    nr, nc = grid_of(len(combos), 5)
    fig, axes = plt.subplots(nr, nc, figsize=(2.9 * nc + 1.6, (0.27 * len(models) + 1.1) * nr + 0.5), sharex=True,
                             sharey=True, squeeze=False, constrained_layout=True)
    handles = []
    for ax, combo in zip(axes.flat, combos):
        for k, area in enumerate(AREAS):
            g = t[(t['combo'] == combo) & (t['area'] == area)].set_index('model').reindex(models)
            off = 0.22 * (0.5 - k)                                     # rural above, urban below
            ax.scatter(g['R2'], yy + off, s=30, marker=AREA_MARKER[area], color=AREA_COLOR[area],
                       edgecolors=SURFACE, linewidths=1.0, zorder=3, label=area)
        ax.set_yticks(yy)
        ax.set_yticklabels([model_label(m) for m in models])
        ax.grid(axis='x')
        ax.set_axisbelow(True)
        ax.set_title(short_combo(combo), fontsize=9)
        handles = ax.get_legend_handles_labels()[0]
    for ax in axes.flat[len(combos):]:
        ax.axis('off')
    axes.flat[0].set_xlim(max(np.nanmin(t['R2']) - 0.03, -1), min(1, np.nanmax(t['R2']) + 0.03))   # below -1: cut
    for ax in axes[-1]:
        ax.set_xlabel('R² within the subset')
    fig.legend(handles=handles, labels=['rural clusters', 'urban clusters'], loc='outside lower center', ncol=2)
    fig.suptitle(f'{start.capitalize()}: R² of the models of 11-15 for rural and urban clusters, whole continent',
                 x=0.01, ha='left', fontsize=11)
    save_figure(fig, f'rural_urban_r2_{start}')


for start in [s for s in START_ORDER if s in set(cont['start'])]:
    r2_area_figure(start)

# 3. Every input against Landsat + NL on the whole continent, rural and urban

For every run with a baseline run (same family, model and start), the paired comparison with it on all, on the rural and on the urban test clusters of the continent, and **`gain rural − gain urban`**: the ΔR² of the rural clusters minus that of the urban clusters, with its interval from the same resamples (`diff low`, `diff high`). A positive difference with an interval above 0 means the input helps rural clusters more than urban ones. `rural_urban_continental_vs_baseline.csv` has every row; the tables show ΔR² [95 %] (▲ better, ▼ worse) per model and input.

In [ ]:
CELLS_CONT = {'all': np.arange(N), **{a: np.flatnonzero(AREA == a) for a in AREAS}}
PAIRS = baseline_pairs(runs)
if not PAIRS:
    print('no run has an input other than Landsat + NL together with its baseline run yet (11, 13, 14, 15; the '
          'other rows of 04/05 and 09/10): the comparisons below stay empty')
rows = []
for run_id, base_id in PAIRS:
    res = compare(Y, P[base_id], P[run_id], CELLS_CONT, BOOT, FOLD_CODE, draws=True)
    dr, du = res['rural'].get('draws'), res['urban'].get('draws')
    diff = {}
    if dr is not None and du is not None and 'dR2' in res['rural'] and 'dR2' in res['urban']:
        b = (dr - du)[np.isfinite(dr - du)]
        if len(b) >= 10:
            diff = {'gain rural - gain urban': res['rural']['dR2'] - res['urban']['dR2'],
                    'diff low': float(np.quantile(b, 0.025)), 'diff high': float(np.quantile(b, 0.975))}
    for area, rec in res.items():
        rows.append({**run_info(run_id), 'area': area, 'baseline run': base_id,
                     **{k: rec.get(k, np.nan) for k in PAIR_COLS}, **(diff if area == 'all' else {})})
cvs = pd.DataFrame(rows, columns=['start', 'family', 'model', 'combo', 'notebook', 'area', 'baseline run'] + PAIR_COLS
                   + ['gain rural - gain urban', 'diff low', 'diff high'])
cvs = cvs.rename(columns={'R2 a': 'R2 baseline', 'R2 b': 'R2'})
cvs = add_q(cvs, ['start', 'family', 'area'], where=cvs['p'].notna(), split=baseline_split(cvs))
cvs['verdict'] = verdict(cvs['dR2 low'], cvs['dR2 high'])
cvs['rural vs urban'] = np.where(cvs['area'] == 'all', verdict(cvs['diff low'], cvs['diff high']), '')
cvs['rural vs urban'] = cvs['rural vs urban'].replace({'better': 'rural gains more', 'worse': 'urban gains more'})
cvs = sort_runs(cvs).reset_index(drop=True)
write_csv(cvs, os.path.join(EVAL_DIR, 'rural_urban_continental_vs_baseline.csv'))
print(f'{len(PAIRS)} runs against their baseline ->', os.path.join(EVAL_DIR, 'rural_urban_continental_vs_baseline.csv'))

for start in [s for s in START_ORDER if s in set(cvs['start'])]:
    for fam in [f for f in FAMILY_ORDER if f in set(cvs.loc[cvs['start'] == start, 'family'])]:
        t = cvs[(cvs['start'] == start) & (cvs['family'] == fam)]
        cell = [fmt_interval(d, lo, hi) + {'better': ' ▲', 'worse': ' ▼'}.get(v, '')
                for d, lo, hi, v in zip(t['dR2'], t['dR2 low'], t['dR2 high'], t['verdict'])]
        m = t.assign(cell=cell).pivot_table(index=['model', 'combo'], columns='area', values='cell', aggfunc='first')
        diff = t[t['area'] == 'all'].set_index(['model', 'combo'])
        m['gain rural - gain urban'] = [fmt_interval(*diff.loc[i, ['gain rural - gain urban', 'diff low', 'diff high']])
                                        if i in diff.index else '' for i in m.index]
        m = m.reindex(columns=[c for c in ['all'] + AREAS + ['gain rural - gain urban'] if c in m.columns])
        m = m.iloc[sorted(range(len(m)), key=lambda i: (model_order(m.index[i][0]), combo_order(m.index[i][1])))]
        m.index = [f'{model_label(a)} | {short_combo(b)}' for a, b in m.index]
        print(f'\n{start.upper()}, {FAMILY_LABEL[fam]}: ΔR² against Landsat + NL on the whole continent [95 %]')
        display(m.fillna(''))

In [ ]:
def forest_area_figure(start):
    t = cvs[(cvs['start'] == start) & (cvs['family'] == 'ten_models') & cvs['area'].isin(AREAS) & cvs['dR2'].notna()]
    if not len(t):
        return
    combos = detail_inputs(t['combo'])
    models = sorted(set(t['model']), key=model_order)
    yy = np.arange(len(models))[::-1]
    nr, nc = grid_of(len(combos), 4)
    fig, axes = plt.subplots(nr, nc, figsize=(3.3 * nc + 1.6, (0.3 * len(models) + 1.1) * nr + 0.6), sharex=True,
                             sharey=True, squeeze=False, constrained_layout=True)
    handles = []
    for ax, combo in zip(axes.flat, combos):
        ax.axvline(0, color=SECONDARY, linewidth=0.8)
        handles = []
        for k, area in enumerate(AREAS):
            g = t[(t['combo'] == combo) & (t['area'] == area)].set_index('model').reindex(models)
            handles.append(forest(ax, yy + 0.2 * (0.5 - k), g['dR2'], g['dR2 low'], g['dR2 high'], AREA_COLOR[area],
                                  AREA_MARKER[area], label=f'{area} clusters'))
        ax.set_yticks(yy)
        ax.set_yticklabels([model_label(m) for m in models])
        ax.grid(axis='x')
        ax.set_axisbelow(True)
        ax.set_title(short_combo(combo), fontsize=9)
    for ax in axes.flat[len(combos):]:
        ax.axis('off')
    for ax in axes[-1]:
        ax.set_xlabel('ΔR² against Landsat + NL')
    fig.legend(handles=handles + FILLED_OPEN, loc='outside lower center', ncol=4)
    fig.suptitle(f'{start.capitalize()}: every input of the models of 11-15 against Landsat + NL, rural and urban clusters, '
                 'whole continent (95 % intervals)', x=0.01, ha='left', fontsize=11)
    save_figure(fig, f'rural_urban_vs_baseline_{start}')


for start in [s for s in START_ORDER if s in set(cvs['start'])]:
    forest_area_figure(start)
summary = cvs[(cvs['area'] != 'all') & ~cvs['model'].isin(SUMMARY_SKIP) & cvs['dR2'].notna()]
if len(summary):
    s = (summary.groupby(['start', 'family', 'combo', 'area'], sort=False)
         .agg(models=('dR2', 'size'), median_dR2=('dR2', 'median'),
              better=('verdict', lambda v: (v == 'better').sum()), worse=('verdict', lambda v: (v == 'worse').sum()))
         .reset_index())
    s = s.pivot_table(index=['start', 'family', 'combo'], columns='area',
                      values=['median_dR2', 'better', 'worse', 'models'], aggfunc='first')
    s = s[[(v, a) for a in AREAS for v in ['models', 'median_dR2', 'better', 'worse'] if (v, a) in s.columns]]
    s.columns = [f"{a} {b.replace('median_dR2', 'median dR2').replace('better', 'sig. better').replace('worse', 'sig. worse')}"
                 for b, a in s.columns]
    s = s.reset_index()
    for b in [b for b in BASELINE_MODELS if b in set(cvs['model'])]:     # every baseline model on its own
        own = cvs[(cvs['model'] == b) & (cvs['area'] != 'all')].pivot_table(index=['start', 'family', 'combo'],
                                                                             columns='area', values='dR2')
        own.columns = [f'{a} {b} dR2' for a in own.columns]
        s = s.merge(own.reset_index(), on=['start', 'family', 'combo'], how='left')
    print('per input, over the models (without', SUMMARY_SKIP, '): median ΔR², models significantly better and worse'
          + ('; the ΔR² of the baseline models on their own' if set(BASELINE_MODELS) & set(cvs['model']) else ''))
    display(sort_runs(s.assign(model='')).drop(columns='model').assign(combo=lambda d: d['combo'].map(short_combo))
            .round(4).reset_index(drop=True))

# 4. Rural and urban clusters in every country

The scores of every run on the rural and on the urban test clusters of every country (`rural_urban_country_metrics.csv`, with `n` and the mean and sd of the observed IWI of the subset). R² needs `MIN_N` clusters, so in countries with few urban clusters the urban R² is missing. The plot shows the R² of the models of 11–15 with Landsat + NL; the table its median over the ten models (without `satclip_only` and the baseline models).

In [ ]:
rows = []
for r in runs.itertuples():
    for area in AREAS:
        code = np.where(AREA == area, COUNTRY_CODE, -1)
        for c, v in zip(COUNTRIES, cell_scores(Y, P[r.run_id], code, len(COUNTRIES))):
            if v[0] > 0:
                rows.append({**run_info(r.run_id), 'country': c, 'area': area, **dict(zip(SCORE_COLS, v)),
                             'run_id': r.run_id})
cm = pd.DataFrame(rows)
write_csv(cm, os.path.join(EVAL_DIR, 'rural_urban_country_metrics.csv'))
print(f'{len(cm)} rows (run x country x area) ->', os.path.join(EVAL_DIR, 'rural_urban_country_metrics.csv'))

base_cm = cm[(cm['family'] == 'ten_models') & (cm['combo'] == BASELINE)]
for start in [s for s in START_ORDER if s in set(base_cm['start'])]:
    t = base_cm[(base_cm['start'] == start) & ~base_cm['model'].isin(SUMMARY_SKIP)]
    m = t.groupby(['country', 'area']).agg(n=('n', 'max'), iwi_mean=(f'{TARGET} mean', 'first'),
                                           iwi_sd=(f'{TARGET} sd', 'first'), median_R2=('R2', 'median'),
                                           median_RMSE=('RMSE', 'median'), median_bias=('bias', 'median'))
    m = m.unstack('area')
    m.columns = [f'{a} {b}' for a, b in m.columns]
    m = m.reindex(index=[c for c in COUNTRIES if c in m.index],
                  columns=[f'{a} {b}' for a in ['n', 'iwi_mean', 'iwi_sd', 'median_R2', 'median_RMSE', 'median_bias']
                           for b in AREAS if f'{a} {b}' in m.columns])
    for area in AREAS:
        if f'n {area}' in m:
            m[f'n {area}'] = m[f'n {area}'].astype('Int64')
    print(f'\n{start.upper()}: the ten models with Landsat + NL (without satclip_only and the baseline models), median '
          'over the models')
    display(m.round(3))

In [ ]:
def r2_area_country_figure():
    t = base_cm[base_cm['model'].isin(TEN_MODELS + BASELINE_MODELS)]
    starts = [s for s in START_ORDER if s in set(t['start'])]
    if not starts:
        return
    models = sorted(set(t['model']), key=model_order)
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    count = {(c, a): int(((COUNTRY == c) & (AREA == a)).sum()) for c in countries for a in AREAS}
    panels = [(s, a) for s in starts for a in AREAS]
    fig, axes = plt.subplots(1, len(panels), figsize=((0.32 * len(models) + 0.3) * len(panels) + 3.2,
                                                      0.24 * len(countries) + 2.6),
                             squeeze=False, sharey=True, constrained_layout=True)
    for ax, (start, area) in zip(axes[0], panels):
        m = t[(t['start'] == start) & (t['area'] == area)].pivot(index='country', columns='model', values='R2')
        m = m.reindex(index=countries, columns=models).to_numpy(dtype=float)
        im = heatmap(ax, m, [f"{c} ({count[(c, 'rural')]} / {count[(c, 'urban')]})" for c in countries],
                     [model_label(k) for k in models], SEQUENTIAL, Normalize(0, 1),
                     labels=[[f'{x:.2f}' if np.isfinite(x) else '' for x in row] for row in m], fontsize=5)
        ax.set_title(f'{start}, {area}', fontsize=9)
    fig.colorbar(im, ax=list(axes[0]), shrink=0.6, extend='min', label='R² within the rural or urban clusters')
    fig.suptitle('R² of the models of 11-15 with Landsat + NL for rural and urban clusters per country '
                 '(rural / urban clusters in brackets)', x=0.01, ha='left', fontsize=11)
    save_figure(fig, 'rural_urban_country_r2_baseline')


r2_area_country_figure()

# 5. Every input against Landsat + NL in every country, rural and urban

The paired comparison of every run with its baseline run on the rural and on the urban clusters of every country (`rural_urban_country_vs_baseline.csv`; `q` over the countries of one model, input, start and area), and over the models (`rural_urban_country_consensus.csv`, without `satclip_only` and the baseline models; `noise dR2` is the ΔR² of `satclip_only`, `vgg16bn dR2` and `mlp dR2` those of the baseline models). The tables: **median ΔR² (models better / models; ▲ significantly better, ▼ worse)** for rural and urban clusters, one table per start and family.

In [ ]:
CELLS_CA = {(c, a): np.flatnonzero((COUNTRY == c) & (AREA == a)) for c in COUNTRIES for a in AREAS}
t0 = time.time()
rows = []
for run_id, base_id in PAIRS:
    for (c, a), rec in compare(Y, P[base_id], P[run_id], CELLS_CA, BOOT, FOLD_CODE).items():
        rows.append({**run_info(run_id), 'country': c, 'area': a, 'baseline run': base_id,
                     **{k: rec.get(k, np.nan) for k in PAIR_COLS}})
ca = pd.DataFrame(rows, columns=['start', 'family', 'model', 'combo', 'notebook', 'country', 'area', 'baseline run']
                  + PAIR_COLS)
ca = ca.rename(columns={'R2 a': 'R2 baseline', 'R2 b': 'R2'})
ca = add_q(ca, ['start', 'family', 'model', 'combo', 'area'], where=ca['p'].notna())
ca['verdict'] = verdict(ca['dR2 low'], ca['dR2 high'])
ca = sort_runs(ca).reset_index(drop=True)
write_csv(ca, os.path.join(EVAL_DIR, 'rural_urban_country_vs_baseline.csv'))
print(f'{len(PAIRS)} runs x {len(CELLS_CA)} country and area cells in {time.time() - t0:.0f} s ->',
      os.path.join(EVAL_DIR, 'rural_urban_country_vs_baseline.csv'))


cons = consensus(ca, ['start', 'family', 'combo', 'country', 'area'])
write_csv(cons, os.path.join(EVAL_DIR, 'rural_urban_country_consensus.csv'))

for start in [s for s in START_ORDER if len(cons) and s in set(cons['start'])]:
    for fam in [f for f in FAMILY_ORDER if f in set(cons.loc[cons['start'] == start, 'family'])]:
        t = cons[(cons['start'] == start) & (cons['family'] == fam)]
        m = t.assign(cell=t.apply(consensus_cell, axis=1)).pivot_table(index='country', columns=['combo', 'area'],
                                                                      values='cell', aggfunc='first')
        cols = [(c, a) for c in sorted(set(t['combo']), key=combo_order) for a in AREAS if (c, a) in m.columns]
        m = m.reindex(index=[c for c in COUNTRIES if c in m.index], columns=cols).fillna('')
        m.columns = [f'{short_combo(c)} | {a}' for c, a in m.columns]
        print(f'\n{start.upper()}, {FAMILY_LABEL[fam]}: median ΔR² against Landsat + NL over the models, rural and urban '
              '(models better / models; ▲ significantly better, ▼ worse)')
        display(m)

# 6. Figures per country, rural and urban

* `rural_urban_country_consensus_<start>.png`: the median ΔR² over the ten models (without `satclip_only` and the baseline models) for every country, input and area, with the number of models that gain.
* `rural_urban_country_<start>_<input>.png`: ΔR² of every model in every country for one input of `DETAIL_INPUTS`, rural and urban side by side; a dot marks an interval that excludes 0.
* `rural_urban_country_<start>_<family>.png` for the Table 2 networks and the ridge: one panel per model and area, every input.

Blue: the input is better than Landsat + NL; red: worse. A dash: fewer than `MIN_N` clusters of that kind in the country, or no run.

In [ ]:
def consensus_area_figure(start):
    t = cons[(cons['start'] == start) & (cons['family'] == 'ten_models')]
    if not len(t):
        return
    combos = sorted(set(t['combo']), key=combo_order)
    cols = [(c, a) for c in combos for a in AREAS]
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    idx = t.set_index(['country', 'combo', 'area'])
    med = np.array([[idx['median dR2'].get((c, k, a), np.nan) for k, a in cols] for c in countries], dtype=float)
    lab = [[(f"{fmt_delta(idx['median dR2'][(c, k, a)])}\n{idx['models better'][(c, k, a)]}/{idx['models'][(c, k, a)]}"
             if (c, k, a) in idx.index and idx['models'][(c, k, a)] and np.isfinite(idx['median dR2'][(c, k, a)])
             else '') for k, a in cols] for c in countries]
    fig, ax = plt.subplots(figsize=(0.62 * len(cols) + 3.6, 0.36 * len(countries) + 2.6), constrained_layout=True)
    im = heatmap(ax, med, [c for c in countries], [f'{short_combo(k)} | {a}' for k, a in cols], DIVERGING,
                 delta_norm(med), labels=lab, fontsize=5)
    ax.xaxis.tick_top()
    ax.tick_params(axis='x', labelrotation=90)
    for j in range(2, len(cols), 2):                                   # a line between the inputs
        ax.axvline(j - 0.5, color=SECONDARY, linewidth=0.8)
    fig.colorbar(im, ax=ax, shrink=0.6, extend='both', label='median ΔR² over the models (input − Landsat + NL)')
    ax.set_title(f'{start.capitalize()}: the ten models, median ΔR² against Landsat + NL per country, rural and urban\n'
                 '(text: median ΔR², models better / models; without satclip_only and the baseline models)', fontsize=10,
                 pad=12)
    save_figure(fig, f'rural_urban_country_consensus_{start}')


def area_detail_figure(start, combo):
    t = ca[(ca['start'] == start) & (ca['family'] == 'ten_models') & (ca['combo'] == combo)]
    if not len(t):
        return
    models = sorted(set(t['model']), key=model_order)
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    norm = delta_norm(ca.loc[(ca['start'] == start) & (ca['family'] == 'ten_models') &
                             ~ca['model'].isin(NOISE_MODELS), 'dR2'])
    fig, axes = plt.subplots(1, 2, figsize=(2 * (0.32 * len(models) + 0.4) + 3.4, 0.24 * len(countries) + 2.6),
                             sharey=True, squeeze=False, constrained_layout=True)
    for ax, area in zip(axes[0], AREAS):
        g = t[t['area'] == area]
        m = g.pivot(index='country', columns='model', values='dR2').reindex(index=countries, columns=models)
        s = g.assign(s=g['verdict'].isin(['better', 'worse']).astype(float)).pivot(
            index='country', columns='model', values='s').reindex(index=countries, columns=models).fillna(0) > 0.5
        im = heatmap(ax, m.to_numpy(dtype=float), countries, [model_label(k) for k in models], DIVERGING, norm,
                     marks=s.to_numpy(dtype=bool), fontsize=5)
        ax.set_title(f'{area} clusters', fontsize=9)
    fig.colorbar(im, ax=list(axes[0]), shrink=0.6, extend='both',
                 label='ΔR² against Landsat + NL (dot: 95 % interval excludes 0)')
    fig.suptitle(f'{start.capitalize()}: {combo_label(combo)} against Landsat + NL, per country', x=0.01, ha='left',
                 fontsize=11)
    save_figure(fig, f"rural_urban_country_{start}_{re.sub(r'[^A-Za-z0-9]+', '_', short_combo(combo)).strip('_')}")


def area_family_figure(start, fam):
    t = ca[(ca['start'] == start) & (ca['family'] == fam)]
    if not len(t):
        return
    models = sorted(set(t['model']), key=model_order)
    combos = sorted(set(t['combo']), key=combo_order)
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    panels = [(m, a) for m in models for a in AREAS]
    norm = delta_norm(t['dR2'])
    nr, nc = grid_of(len(panels), 4)
    fig, axes = plt.subplots(nr, nc, figsize=((0.32 * len(combos) + 0.4) * nc + 3.2,
                                              (0.24 * len(countries) + 1.8) * nr + 0.5),
                             sharey=True, squeeze=False, constrained_layout=True)
    for ax, (model, area) in zip(axes.flat, panels):
        g = t[(t['model'] == model) & (t['area'] == area)]
        m = g.pivot(index='country', columns='combo', values='dR2').reindex(index=countries, columns=combos)
        s = g.assign(s=g['verdict'].isin(['better', 'worse']).astype(float)).pivot(
            index='country', columns='combo', values='s').reindex(index=countries, columns=combos).fillna(0) > 0.5
        im = heatmap(ax, m.to_numpy(dtype=float), countries, [short_combo(c) for c in combos], DIVERGING, norm,
                     marks=s.to_numpy(dtype=bool), fontsize=5)
        ax.set_title(f'{model_label(model)}, {area}', fontsize=9)
    for ax in axes.flat[len(panels):]:
        ax.axis('off')
    fig.colorbar(im, ax=list(axes.flat), shrink=0.5, extend='both',
                 label='ΔR² against Landsat + NL (dot: 95 % interval excludes 0)')
    fig.suptitle(f'{start.capitalize()}: {FAMILY_LABEL[fam]}, every row against Landsat + NL, rural and urban, '
                 'per country', x=0.01, ha='left', fontsize=11)
    save_figure(fig, f'rural_urban_country_{start}_{fam}')


for start in [s for s in START_ORDER if s in set(ca['start'])]:
    consensus_area_figure(start)
    for combo in detail_inputs(ca.loc[(ca['start'] == start) & (ca['family'] == 'ten_models'), 'combo']):
        area_detail_figure(start, combo)
    for fam in ('table2', 'ridge'):
        area_family_figure(start, fam)

# 7. Every model against the baseline models, rural and urban

The baseline models of 11–15 (`BASELINE_MODELS`: VGG-16-BN and the MLP on band statistics) as the reference for every model of 11–15 on the rural and on the urban clusters, with the same start, in the two ways of 16: with the **same input** (ΔR² = R²(model with input I) − R²(baseline model with input I): what the architecture adds) and against the baseline model on **Landsat + NL** (what the architecture and the added bands add together; for the baseline model itself its own gain from the input).

* **Whole continent** (`rural_urban_continental_vs_baseline_models.csv`; `q` over the models and inputs of a start, baseline model, reference and area), with `gain rural − gain urban` and its interval as in section 3: a positive difference with an interval above 0 means the model's lead over the baseline model is larger on rural clusters. The table: the summary over the ten models (without `satclip_only` and the baseline models; `rural_urban_continental_vs_baseline_models_summary.csv`). The plot `rural_urban_vs_<baseline model>_<start>.png`: every model and input (Landsat + NL and `DETAIL_INPUTS`), rural and urban; top row same input, bottom row against the baseline model on Landsat + NL (`satclip_only`, which reads no band and lies far below, is left out of the plot).
* **Per country** (`rural_urban_country_vs_baseline_models.csv`; `q` over the countries of one model, input, start, baseline model, reference and area) and over the ten models (`rural_urban_country_vs_baseline_models_summary.csv`). The tables: **median ΔR² over the ten models (models better / models; ▲ significantly better, ▼ worse)** for rural and urban clusters, one per start, baseline model and reference; the plot `rural_urban_country_vs_baseline_models_<start>_<baseline model>.png` (left: same input, right: on Landsat + NL).

In [ ]:
def area_difference(res):
    """gain rural - gain urban and its interval from the paired resamples of one comparison (compare(..., draws=True))."""
    dr, du = res['rural'].get('draws'), res['urban'].get('draws')
    if dr is None or du is None or 'dR2' not in res['rural'] or 'dR2' not in res['urban']:
        return {}
    d = (dr - du)[np.isfinite(dr - du)]
    if len(d) < 10:
        return {}
    return {'gain rural - gain urban': res['rural']['dR2'] - res['urban']['dR2'],
            'diff low': float(np.quantile(d, 0.025)), 'diff high': float(np.quantile(d, 0.975))}


BM_PAIRS = baseline_model_pairs(runs) if VS_BASELINE_MODELS else []
t0 = time.time()
rows = []
for run_id, ref_id, bm, against in BM_PAIRS:
    res = compare(Y, P[ref_id], P[run_id], CELLS_CONT, BOOT, FOLD_CODE, draws=True)
    diff = area_difference(res)
    for area, rec in res.items():
        rows.append({**run_info(run_id), 'baseline model': bm, 'against': against, 'area': area, 'baseline run': ref_id,
                     **{k: rec.get(k, np.nan) for k in PAIR_COLS}, **(diff if area == 'all' else {})})
cbm = pd.DataFrame(rows, columns=['start', 'family', 'model', 'combo', 'notebook', 'baseline model', 'against', 'area',
                                  'baseline run'] + PAIR_COLS + ['gain rural - gain urban', 'diff low', 'diff high'])
cbm = cbm.rename(columns={'R2 a': 'R2 baseline model', 'R2 b': 'R2'})
cbm = add_q(cbm, ['start', 'baseline model', 'against', 'area'], where=cbm['p'].notna())
cbm['verdict'] = verdict(cbm['dR2 low'], cbm['dR2 high'])
cbm['rural vs urban'] = np.where(cbm['area'] == 'all', verdict(cbm['diff low'], cbm['diff high']), '')
cbm['rural vs urban'] = cbm['rural vs urban'].replace({'better': 'rural gains more', 'worse': 'urban gains more'})
cbm = sort_bm(cbm)
cbm_sum = bm_summary(cbm, ['combo', 'area'])
if VS_BASELINE_MODELS:
    write_csv(cbm, os.path.join(EVAL_DIR, 'rural_urban_continental_vs_baseline_models.csv'))
    write_csv(cbm_sum, os.path.join(EVAL_DIR, 'rural_urban_continental_vs_baseline_models_summary.csv'))
    print(f'{len(BM_PAIRS)} comparisons with the baseline models in {time.time() - t0:.0f} s ->',
          os.path.join(EVAL_DIR, 'rural_urban_continental_vs_baseline_models.csv'))
if not len(cbm):
    print('no comparison with the baseline models: ' + ('VS_BASELINE_MODELS = False' if not VS_BASELINE_MODELS else
          'no baseline model (BASELINE_MODELS) among the runs of 11-15 yet (run 11-15 with --array=10-11)'))

for start in [s for s in START_ORDER if len(cbm_sum) and s in set(cbm_sum['start'])]:
    t = cbm_sum[(cbm_sum['start'] == start) & cbm_sum['area'].isin(AREAS)]
    if not len(t):
        continue
    t = t.assign(cell=t.apply(consensus_cell, axis=1),
                 row=[f'minus {bm_label(b, a)}' for b, a in zip(t['baseline model'], t['against'])])
    m = t.pivot_table(index='row', columns=['combo', 'area'], values='cell', aggfunc='first')
    cols = [(c, a) for c in sorted(set(t['combo']), key=combo_order) for a in AREAS if (c, a) in m.columns]
    m = m.reindex(index=list(dict.fromkeys(t['row'])), columns=cols).fillna('')
    m.columns = [f'{short_combo(c)} | {a}' for c, a in m.columns]
    print(f'\n{start.upper()}: the ten models minus the baseline models on the rural and urban clusters of the whole '
          'continent, median ΔR² (models better / models; ▲ significantly better, ▼ worse)')
    display(m)


def bm_forest_area_figure(start, b):
    t = cbm[(cbm['start'] == start) & (cbm['baseline model'] == b) & cbm['area'].isin(AREAS) & cbm['dR2'].notna()
            & ~cbm['model'].isin(NOISE_MODELS)]
    refs = [a for a in BM_AGAINST if a in set(t['against'])]
    if not refs:
        return
    have = set(t['combo'])
    combos = sorted(have, key=combo_order) if DETAIL_INPUTS == 'all' else \
        [c for c in [BASELINE] + MAIN_INPUTS if c in have] or sorted(have, key=combo_order)
    models = sorted(set(t['model']), key=model_order)
    yy = np.arange(len(models))[::-1]
    fig, axes = plt.subplots(len(refs), len(combos), figsize=(3.0 * len(combos) + 1.8,
                                                              (0.3 * len(models) + 1.0) * len(refs) + 0.9),
                             sharex=True, sharey=True, squeeze=False, constrained_layout=True)
    handles = []
    for i, against in enumerate(refs):
        for j, combo in enumerate(combos):
            ax = axes[i][j]
            ax.axvline(0, color=SECONDARY, linewidth=0.8)
            handles = []
            for k, area in enumerate(AREAS):
                g = t[(t['against'] == against) & (t['combo'] == combo) & (t['area'] == area)].set_index('model')
                g = g.reindex(models)
                handles.append(forest(ax, yy + 0.2 * (0.5 - k), g['dR2'], g['dR2 low'], g['dR2 high'],
                                      AREA_COLOR[area], AREA_MARKER[area], label=f'{area} clusters'))
            ax.set_yticks(yy)
            ax.set_yticklabels([model_label(m) for m in models])
            ax.grid(axis='x')
            ax.set_axisbelow(True)
            ax.set_title(f"{short_combo(combo)} | {'same input' if against == 'same input' else 'baseline on Landsat + NL'}",
                         fontsize=8.5)
    for ax in axes[-1]:
        ax.set_xlabel(f'ΔR² against {model_label(b)}')
    fig.legend(handles=handles + FILLED_OPEN, loc='outside lower center', ncol=4)
    fig.suptitle(f'{start.capitalize()}: every model of 11-15 minus {model_label(b)}, rural and urban clusters, whole '
                 f'continent (top: same input; bottom: {model_label(b)} on Landsat + NL; 95 % intervals)',
                 x=0.01, ha='left', fontsize=11)
    save_figure(fig, f'rural_urban_vs_{b}_{start}')


for start in [s for s in START_ORDER if len(cbm) and s in set(cbm['start'])]:
    for b in [b for b in BASELINE_MODELS if b in set(cbm.loc[cbm['start'] == start, 'baseline model'])]:
        bm_forest_area_figure(start, b)

In [ ]:
t0 = time.time()
rows = []
for run_id, ref_id, bm, against in BM_PAIRS:
    for (c, a), rec in compare(Y, P[ref_id], P[run_id], CELLS_CA, BOOT, FOLD_CODE).items():
        rows.append({**run_info(run_id), 'baseline model': bm, 'against': against, 'country': c, 'area': a,
                     'baseline run': ref_id, **{k: rec.get(k, np.nan) for k in PAIR_COLS}})
cabm = pd.DataFrame(rows, columns=['start', 'family', 'model', 'combo', 'notebook', 'baseline model', 'against',
                                   'country', 'area', 'baseline run'] + PAIR_COLS)
cabm = cabm.rename(columns={'R2 a': 'R2 baseline model', 'R2 b': 'R2'})
cabm = add_q(cabm, ['start', 'model', 'combo', 'baseline model', 'against', 'area'], where=cabm['p'].notna())
cabm['verdict'] = verdict(cabm['dR2 low'], cabm['dR2 high'])
cabm = sort_bm(cabm)
cabm_sum = bm_summary(cabm, ['combo', 'country', 'area'])
if VS_BASELINE_MODELS:
    write_csv(cabm, os.path.join(EVAL_DIR, 'rural_urban_country_vs_baseline_models.csv'))
    write_csv(cabm_sum, os.path.join(EVAL_DIR, 'rural_urban_country_vs_baseline_models_summary.csv'))
    print(f'{len(BM_PAIRS)} comparisons x {len(CELLS_CA)} country and area cells in {time.time() - t0:.0f} s ->',
          os.path.join(EVAL_DIR, 'rural_urban_country_vs_baseline_models.csv'))

for start in [s for s in START_ORDER if len(cabm_sum) and s in set(cabm_sum['start'])]:
    for b in [b for b in BASELINE_MODELS if b in set(cabm_sum.loc[cabm_sum['start'] == start, 'baseline model'])]:
        for against in BM_AGAINST:
            t = cabm_sum[(cabm_sum['start'] == start) & (cabm_sum['baseline model'] == b) &
                         (cabm_sum['against'] == against)]
            if not len(t):
                continue
            m = t.assign(cell=t.apply(consensus_cell, axis=1)).pivot_table(index='country', columns=['combo', 'area'],
                                                                          values='cell', aggfunc='first')
            cols = [(c, a) for c in sorted(set(t['combo']), key=combo_order) for a in AREAS if (c, a) in m.columns]
            m = m.reindex(index=[c for c in COUNTRIES if c in m.index], columns=cols).fillna('')
            m.columns = [f'{short_combo(c)} | {a}' for c, a in m.columns]
            print(f'\n{start.upper()}: the ten models minus {bm_label(b, against)}, median ΔR² per country, rural and '
                  'urban (models better / models; ▲ significantly better, ▼ worse)')
            display(m)


def bm_area_country_figure(start, b):
    t = cabm_sum[(cabm_sum['start'] == start) & (cabm_sum['baseline model'] == b)]
    refs = [a for a in BM_AGAINST if a in set(t['against'])]
    if not refs:
        return
    combos = sorted(set(t['combo']), key=combo_order)
    cols = [(c, a) for c in combos for a in AREAS]
    countries = [c for c in COUNTRIES if c in set(t['country'])]
    norm = delta_norm(t['median dR2'], floor=0.02)
    fig, axes = plt.subplots(1, len(refs), figsize=((0.62 * len(cols) + 0.5) * len(refs) + 3.6,
                                                    0.36 * len(countries) + 2.8),
                             squeeze=False, sharey=True, constrained_layout=True)
    for ax, against in zip(axes[0], refs):
        idx = t[t['against'] == against].set_index(['country', 'combo', 'area'])
        med = np.array([[idx['median dR2'].get((c, k, a), np.nan) for k, a in cols] for c in countries], dtype=float)
        lab = [[(f"{fmt_delta(idx['median dR2'][(c, k, a)])}\n{idx['models better'][(c, k, a)]}/"
                 f"{idx['models'][(c, k, a)]}"
                 if (c, k, a) in idx.index and idx['models'][(c, k, a)] and np.isfinite(idx['median dR2'][(c, k, a)])
                 else '') for k, a in cols] for c in countries]
        im = heatmap(ax, med, countries, [f'{short_combo(k)} | {a}' for k, a in cols], DIVERGING, norm, labels=lab,
                     fontsize=5)
        ax.xaxis.tick_top()
        ax.tick_params(axis='x', labelrotation=90)
        for j in range(2, len(cols), 2):                               # a line between the inputs
            ax.axvline(j - 0.5, color=SECONDARY, linewidth=0.8)
        ax.set_title(f'minus {bm_label(b, against)}', fontsize=10, pad=8)
    fig.colorbar(im, ax=list(axes[0]), shrink=0.6, extend='both',
                 label=f'median ΔR² over the ten models (model − {model_label(b)})')
    fig.suptitle(f'{start.capitalize()}: the ten models against {model_label(b)} per country, rural and urban (text: '
                 'median ΔR², models better / models; without satclip_only and the baseline models)',
                 x=0.01, ha='left', fontsize=11)
    save_figure(fig, f'rural_urban_country_vs_baseline_models_{start}_{b}')


for start in [s for s in START_ORDER if len(cabm_sum) and s in set(cabm_sum['start'])]:
    for b in [b for b in BASELINE_MODELS if b in set(cabm_sum.loc[cabm_sum['start'] == start, 'baseline model'])]:
        bm_area_country_figure(start, b)